In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# ניטור או ביטול משרה

צפה ברשימת עומסי העבודה שלך ב[דף עומסי העבודה](https://quantum.cloud.ibm.com/workloads).

## צפייה בסטטוס משרה
עבור ל[טבלת עומסי העבודה](https://quantum.cloud.ibm.com/workloads) שלך ובדוק תחת עמודת הסטטוס אם משרה הושלמה או נכשלה.

## צפייה בשימוש הנותר
עבור ל[טבלת ה-Instances](https://quantum.cloud.ibm.com/instances) שלך ובחר את הלשונית המשויכת לתוכנית שברצונך לצפות בשימוש הנותר לה. הזמן הכולל שנוצל והזמן הכולל שנותר בתוכנית שלך מוצגים.

## צפייה במדדים על מספר המשרות ועומסי העבודה שהוגשו
עבור ל[דף Analytics](https://quantum.cloud.ibm.com/analytics) כדי לראות את המספר הכולל של המשרות שהוגשו, כמו גם ספירה של עומסי עבודה בצווארי בקבוק ועומסי עבודה בסשנים. שים לב שניתן לראות את דף Analytics רק עבור חשבונות שבבעלותך או שאתה מנהל.

## ניטור משרה
השתמש ב-instance של המשרה כדי לבדוק את סטטוס המשרה או לאחזר את התוצאות על ידי קריאה לפקודה המתאימה:

|                               |                                                                                                                                                                                                              |
| ----------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| job.result()                  | עיין בתוצאות המשרה מיד לאחר השלמתה. תוצאות המשרה זמינות לאחר השלמת המשרה. לפיכך, job.result() היא קריאה חוסמת עד להשלמת המשרה.                               |
| job.job\_id()                 | החזר את המזהה הייחודי של אותה משרה. אחזור תוצאות המשרה בשלב מאוחר יותר דורש את מזהה המשרה. לפיכך, מומלץ לשמור את המזהים של משרות שייתכן שתרצה לאחזר מאוחר יותר. |
| job.status()                  | בדוק את סטטוס המשרה.                                                                                                                                                                                        |
| job = service.job(\<job\_id>) | אחזר משרה שהגשת בעבר. קריאה זו דורשת את מזהה המשרה.                                                                                                                                      |

<span id="retrieve-later"></span>
## אחזור תוצאות משרה בשלב מאוחר יותר
קרא ל-`service.job(\<job\_id>)` כדי לאחזר משרה שהגשת בעבר. אם אין לך את מזהה המשרה, או אם ברצונך לאחזר מספר משרות בבת אחת; כולל משרות מ-QPUs (יחידות עיבוד קוונטי) שפרשו, קרא ל-`service.jobs()` עם פילטרים אופציונליים במקום. ראה [QiskitRuntimeService.jobs](../api/qiskit-ibm-runtime/qiskit-runtime-service#jobs).

> **Note:** `service.jobs()` מחזיר גם משרות שהורצו מחבילת `qiskit-ibm-provider` המיושנת. משרות שהוגשו על ידי חבילת `qiskit-ibmq-provider` הישנה יותר (גם כן מיושנת) אינן זמינות עוד.

### דוגמה
דוגמה זו מחזירה את 10 משרות ה-runtime האחרונות שהורצו על `my_backend`:

In [1]:
# This cell is hidden from users
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.transpiler import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
import numpy as np


my_backend = "ibm_torino"
service = QiskitRuntimeService()
# backend = service.backend(my_backend)
backend = service.least_busy()

# Define two circuits, each with one parameter with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.cx(0, 1)
circuit.h(0)
circuit.measure_all()


pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)

params = np.random.uniform(size=(2, 3)).T

sampler_pub = (transpiled_circuit, params)

# Instantiate the new estimator object, then run the transpiled circuit
# using the set of parameters and observables.
sampler = SamplerV2(mode=backend)
job = sampler.run([sampler_pub], shots=4)
print(job.job_id())

d305ck0ocacs73ajagvg


In [2]:
result = job.result()


spans = job.result().metadata["execution"]["execution_spans"]
print(spans)

ExecutionSpans([DoubleSliceSpan(<start='2025-09-09 16:31:16', stop='2025-09-09 16:31:16', size=24>)])


In [3]:
params = np.random.uniform(size=(2, 3))
params

array([[0.2260416 , 0.8747859 , 0.44361995],
       [0.94700856, 0.96826017, 0.98426562]])

In [4]:
mask = spans[0].mask(0)
mask

array([[[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]]])

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

# Initialize the account first.
service = QiskitRuntimeService()
# Use `limit` to retrieve a specific number of jobs. The default `limit` is 10.
service.jobs(backend_name=my_backend)

## Cancel a job

You can cancel a job from the IBM Quantum Platform dashboard either on the Workloads page or the details page for a specific workload. On the Workloads page, click the overflow menu at the end of the row for that workload, and select Cancel. If you are on the details page for a specific workload, use the Actions dropdown at the top of the page, and select Cancel.

In Qiskit, use `job.cancel()` to cancel a job.

<span id="execution-spans"></span>
## View Sampler execution spans

The results of [`SamplerV2`](/docs/api/qiskit-ibm-runtime/sampler-v2) jobs executed in Qiskit Runtime contain execution timing information in their metadata.
This timing information can be used to place upper and lower timestamp bounds on when particular shots were executed on the QPU.
Shots are grouped into [`ExecutionSpan`](/docs/api/qiskit-ibm-runtime/execution-span-execution-span) objects, each of which indicates a start time, a stop time, and a specification of which shots were collected in the span.

An execution span specifies which data was executed during its window by providing an [`ExecutionSpan.mask`](/docs/api/qiskit-ibm-runtime/execution-span-execution-span#mask) method. This method, given any [Primitive Unified Block (PUB)](/docs/guides/primitive-input-output) index, returns a boolean mask that is `True` for all shots executed during its window. PUBs are indexed by the order in which they were given to the Sampler run call. If, for example, a PUB has shape `(2, 3)` and was run with four shots, then the mask's shape is `(2, 3, 4)`. See the [execution_span](/docs/api/qiskit-ibm-runtime/execution-span) API page for full details.

Example:

To view execution span information, review the metadata of the result returned by `SamplerV2`, which comes in the form of an `ExecutionSpans` object. This object is a list-like container containing instances of subclasses of `ExecutionSpan`, such as `SliceSpan`:

In [6]:
from qiskit.primitives import BitArray

# Get the mask of the 1st PUB for the 0th span.
mask = spans[0].mask(0)

# Decide whether the 0th shot of parameter set (1, 2) occurred in this span.
in_this_span = mask[2, 1, 0]

# Create a new bit array containing only the PUB-1 data collected during this span.
bits = result[0].data.meas
filtered_data = BitArray(bits.array[mask], bits.num_bits)

Execution spans can be filtered to include information pertaining to specific PUBs, selected by their indices:

In [7]:
# take the subset of spans that reference data in PUBs 0 or 2
spans.filter_by_pub([0, 2])

ExecutionSpans([DoubleSliceSpan(<start='2025-09-09 16:31:16', stop='2025-09-09 16:31:16', size=24>)])

View global information about the collection of execution spans:

In [8]:
print("Number of execution spans:", len(spans))
print("  Start of the first span:", spans.start)
print("     End of the last span:", spans.stop)
print("       Total duration (s):", spans.duration)

Number of execution spans: 1
  Start of the first span: 2025-09-09 16:31:16.320568
     End of the last span: 2025-09-09 16:31:16.865858
       Total duration (s): 0.54529


Extract and inspect a particular span:

In [9]:
spans.sort()
print(" Start of first span:", spans[0].start)
print("   End of first span:", spans[0].stop)
print("#shots in first span:", spans[0].size)

 Start of first span: 2025-09-09 16:31:16.320568
   End of first span: 2025-09-09 16:31:16.865858
#shots in first span: 24


## ביטול משרה
ניתן לבטל משרה מלוח המחוונים של IBM Quantum Platform, בדף עומסי העבודה או בדף הפרטים של עומס עבודה ספציפי. בדף עומסי העבודה, לחץ על תפריט הגלישה בקצה השורה של אותו עומס עבודה, ובחר ביטול. אם אתה נמצא בדף הפרטים של עומס עבודה ספציפי, השתמש בתפריט הנפתח פעולות בראש הדף, ובחר ביטול.

ב-Qiskit, השתמש ב-`job.cancel()` כדי לבטל משרה.

<span id="execution-spans"></span>
## צפייה בטווחי ביצוע של Sampler
תוצאות משרות [`SamplerV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/sampler-v2) המבוצעות ב-Qiskit Runtime מכילות מידע תזמון ביצוע במטא-נתונים שלהן.
מידע תזמון זה ניתן לשימוש כדי למקם חסמי חותמת זמן עליונה ותחתונה על מתי shots מסוימים בוצעו על ה-QPU.
Shots מקובצים לאובייקטי [`ExecutionSpan`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/execution-span-execution-span), שכל אחד מהם מציין זמן התחלה, זמן סיום ומפרט אילו shots נאספו בטווח.

טווח ביצוע מציין אילו נתונים בוצעו במהלך חלונו על ידי מתן שיטת [`ExecutionSpan.mask`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/execution-span-execution-span#mask). שיטה זו, בהינתן אינדקס [Primitive Unified Block (PUB)](/guides/primitive-input-output) כלשהו, מחזירה מסכה בוליאנית שהיא `True` עבור כל ה-shots שבוצעו במהלך חלונה. PUBs מואצפנים לפי הסדר שבו ניתנו לקריאת ריצת Sampler. אם, לדוגמה, ל-PUB יש צורה `(2, 3)` והורץ עם ארבעה shots, אז צורת המסכה היא `(2, 3, 4)`. ראה את דף ה-API [execution_span](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/execution-span) לפרטים מלאים.

דוגמה:
כדי לצפות במידע טווח ביצוע, עיין במטא-נתונים של התוצאה שמוחזרת על ידי `SamplerV2`, שמגיעה בצורת אובייקט `ExecutionSpans`. אובייקט זה הוא מיכל דמוי-רשימה המכיל instances של מחלקות משנה של `ExecutionSpan`, כגון `SliceSpan`: